# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import json
import os
import chromadb

from pprint import pformat
from dotenv import load_dotenv
from tavily import TavilyClient
from pydantic import BaseModel

from lib.agents import Agent
from lib.llm import LLM
from lib.tooling import tool

# Load environment variables
from pathlib import Path
load_dotenv(Path.cwd().parent.parent / "config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Connect to the existing ChromaDB from Part 1
embedding_fn = chromadb.utils.embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-3-large"
)

chroma_client = chromadb.PersistentClient(path="chroma_db")

collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [3]:
class EvaluationReport(BaseModel):
    useful: bool
    description: str

In [4]:
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("TAVILY_API_KEY loaded:", bool(os.getenv("TAVILY_API_KEY")))
print("CHROMA_OPENAI_API_KEY loaded:", bool(os.getenv("CHROMA_OPENAI_API_KEY")))

OPENAI_API_KEY loaded: True
TAVILY_API_KEY loaded: True
CHROMA_OPENAI_API_KEY loaded: True


In [5]:
def print_section(title: str, separator: str = "="):
    print(separator * 52)
    print(title)
    print(separator * 52)


def parse_evaluation_result(evaluation):
    if isinstance(evaluation, EvaluationReport):
        return evaluation

    if isinstance(evaluation, str):
        try:
            evaluation = json.loads(evaluation)
        except json.JSONDecodeError:
            return EvaluationReport(useful=False, description=evaluation)

    if isinstance(evaluation, dict):
        return EvaluationReport(**evaluation)

    return EvaluationReport(
        useful=False,
        description="Unable to parse retrieval evaluation result."
    )


def build_reasoning(trace: dict):
    reasoning = [
        "Searched internal vector database.",
        "Evaluated retrieved documents."
    ]

    if trace["evaluation"]["useful"]:
        reasoning.append("Internal documents were sufficient.")
    else:
        reasoning.append("Internal documents were insufficient.")
        reasoning.append("Used Tavily web search.")

    return reasoning


def run_research_workflow(question: str):
    print_section("STEP 1 - Internal Retrieval")
    internal_docs = retrieve_game(question)
    print("-" * 50)
    print("TOOL USED: retrieve_game")
    print("-" * 50)
    print("RETRIEVED DOCUMENTS")
    print(pformat(internal_docs, sort_dicts=False))
    print()

    print_section("STEP 2 - Evaluation")
    evaluation = parse_evaluation_result(evaluate_retrieval(question, internal_docs))
    print("-" * 50)
    print("TOOL USED: evaluate_retrieval")
    print("-" * 50)
    print("EVALUATION")
    print(pformat(evaluation.model_dump(), sort_dicts=False))
    print()

    print_section("STEP 3 - Web Search")
    if evaluation.useful:
        web_results = []
        print("-" * 50)
        print("TOOL USED: game_web_search")
        print("-" * 50)
        print("WEB RESULTS")
        print("Web search not needed. Internal documents were sufficient.")
        print()
    else:
        web_results = game_web_search(question)
        print("-" * 50)
        print("TOOL USED: game_web_search")
        print("-" * 50)
        print("WEB RESULTS")
        print(pformat(web_results, sort_dicts=False))
        print()

    citations = [result["url"] for result in web_results] if web_results else ["Internal Game Database"]

    return {
        "question": question,
        "internal_docs": internal_docs,
        "evaluation": evaluation.model_dump(),
        "web_results": web_results,
        "citations": citations
    }

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
@tool
def retrieve_game(query: str):
    """
    Semantic search: Finds the most relevant game information in the vector database.

    Args:
        query: A question about the game industry.

    Returns:
        A list of the most relevant games including platform, publisher,
        release year, genre and description.
    """

    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    retrieved_games = []

    for metadata in results["metadatas"][0]:
        retrieved_games.append(metadata)

    return retrieved_games

In [7]:
retrieve_game("Who published Minecraft?")

[{'Genre': 'Sandbox, Survival', 'Description': 'A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adventure.', 'Platform': 'Xbox One', 'Publisher': 'Mojang Studios', 'YearOfRelease': 2014, 'Name': 'Minecraft'}, {'Platform': 'Xbox Series X|S', 'YearOfRelease': 2021, 'Name': 'Halo Infinite', 'Genre': 'First-person shooter', 'Publisher': 'Xbox Game Studios', 'Description': "The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting."}, {'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", 'Platform': 'Nintendo 64', 'Publisher': 'Nintendo', 'Name': 'Super Mario 64', 'Genre': 'Platformer', 'YearOfRelease': 1996}]

#### Evaluate Retrieval Tool

In [8]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: list):
    """
    Based on the user's question and the retrieved documents,
    evaluate whether the retrieved documents are sufficient
    to answer the user's question.

    Args:
        question: Original user question.
        retrieved_docs: Documents retrieved from the Vector Database.

    Returns:
        EvaluationReport containing:
        - useful: whether the documents are sufficient
        - description: explanation of the decision
    """

    llm = LLM(
        model="gpt-4o-mini",
        temperature=0
    )

    prompt = f"""
You are an AI evaluator.

Your task is to determine whether the retrieved documents contain enough
information to answer the user's question.

User Question:
{question}

Retrieved Documents:
{retrieved_docs}

Return:
- useful = true if the documents are sufficient.
- useful = false if more information from the web is needed.

Explain your reasoning briefly.
"""

    result = llm.invoke(
        prompt,
        response_format=EvaluationReport
    )

    return result.content

In [9]:
docs = retrieve_game("Who published Minecraft?")

evaluation = evaluate_retrieval(
    "Who published Minecraft?",
    docs
)

print(evaluation)

{"useful":true,"description":"The retrieved documents contain sufficient information to answer the user's question about who published Minecraft. The first document explicitly states that 'Mojang Studios' is the publisher of Minecraft."}


#### Game Web Search Tool

In [10]:
@tool
def game_web_search(question: str):
    """
    Search the web for game-related information.

    Args:
        question: A question about the video game industry.

    Returns:
        Relevant web search results with citations.
    """

    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=3
    )

    return [
        {
            "title": r["title"],
            "content": r["content"],
            "url": r["url"]
        }
        for r in response["results"]
    ]

In [11]:
game_web_search(
    "Was Mortal Kombat X released for PlayStation 5?"
)

[{'title': 'Mortal Kombat X', 'content': 'An upgraded version of Mortal Kombat X, titled Mortal Kombat XL, was released on March 1, 2016, for PlayStation 4 and Xbox One, including all downloadable content characters from the two released Kombat Packs, almost all bonus alternate costumes available at the time of release, improved gameplay, and improved netcode. This edition was also released for Windows on October 4, 2016. A sequel, Mortal Kombat 11, was released on April 23, 2019, for Nintendo Switch, PlayStation 4, Windows, and Xbox One. [...] ### Mortal Kombat XL\n\n[edit]\n\nOn January 20, 2016, coinciding with the official announcement of Kombat Pack 2, NetherRealm Studios announced a new release of the game, called Mortal Kombat XL to be released simultaneously with said Kombat Pack, which would include all previously released downloadable content up to Kombat Pack 2. Mortal Kombat XL was then released for Xbox One and PlayStation 4 on March 1, 2016, in North America, and March 4,

### Agent

In [12]:
instructions = """
You are UdaPlay, an AI Research Agent specializing in video games.

Workflow:
1. Always begin by calling retrieve_game to search the internal game database.
2. Always call evaluate_retrieval to determine whether the retrieved documents are sufficient.
3. If evaluate_retrieval indicates the documents are not useful, call game_web_search.
4. Prefer information from the internal database whenever possible.
5. Use web search only when necessary.
6. When web search is used, include the relevant source URLs in your answer.
7. Present answers clearly using bullet points whenever appropriate.
8. If previous conversation context is relevant, use it naturally.
9. Never fabricate information. If neither the database nor the web contains sufficient information, clearly state that.
"""
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search
    ],
    temperature=0
)

In [13]:
queries = [
    "When Pokemon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?"
]

for index, query in enumerate(queries, start=1):
    print_section("QUESTION")
    print(query)
    print()

    trace = run_research_workflow(query)

    print_section("REASONING", "-")
    for step in build_reasoning(trace):
        print(f"- {step}")
    print()

    run = agent.invoke(query, session_id=f"demo-{index}")
    final_state = run.get_final_state()

    print_section("FINAL ANSWER", "-")
    print(final_state["messages"][-1].content)
    print()

    print_section("CITATIONS", "-")
    for citation in trace["citations"]:
        print(f"- {citation}")
    print()

QUESTION
When Pokemon Gold and Silver was released?

STEP 1 - Internal Retrieval
--------------------------------------------------
TOOL USED: retrieve_game
--------------------------------------------------
RETRIEVED DOCUMENTS
[{'YearOfRelease': 1999,
  'Description': 'Second-generation Pokémon games introducing new regions, '
                 'Pokémon, and gameplay mechanics.',
  'Name': 'Pokémon Gold and Silver',
  'Genre': 'Role-playing',
  'Publisher': 'Nintendo',
  'Platform': 'Game Boy Color'},
 {'YearOfRelease': 2002,
  'Description': 'Third-generation Pokémon games set in the Hoenn region, '
                 'featuring new Pokémon and double battles.',
  'Publisher': 'Nintendo',
  'Name': 'Pokémon Ruby and Sapphire',
  'Platform': 'Game Boy Advance',
  'Genre': 'Role-playing'},
 {'Genre': 'Platformer',
  'Description': 'A groundbreaking 3D platformer that set new standards for '
                 "the genre, featuring Mario's quest to rescue Princess Peach.",
  'Publisher': 'Ni

### (Optional) Advanced

In [14]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes